# CenterSpeed 학습 (WandB로 train/val loss 시각화)
- 이 노트북은 train loss, val loss를 wandb로 실시간 시각화합니다.
- 나머지 기능(데이터셋, 모델, loss 등)은 기존과 동일하게 유지됩니다.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys, os, random, datetime
from torch.utils.data import DataLoader, random_split
import wandb

current_dir = os.path.dirname(os.path.abspath(''))
two_up_dir = os.path.dirname(os.path.dirname(current_dir))
sys.path.append(two_up_dir)

# from TinyCenterSpeed.src.models.resnet import *
from TinyCenterSpeed.src.models.CenterSpeed import *
from TinyCenterSpeed.dataset.CenterSpeed_dataset import *
from TinyCenterSpeed.src.models.losses import *
from train import *



%env "WANDB_NOTEBOOK_NAME" "train_CenterSpeed_wandb.ipynb"
wandb.login()

env: "WANDB_NOTEBOOK_NAME"="train_CenterSpeed_wandb.ipynb"


wandb: Currently logged in as: whdaudpark (whdaudpark-dongguk-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
transform = transforms.Compose([RandomRotation(45),
                                RandomFlip(0.5)])


try:
    set = CenterSpeedDataset('/home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT', transform=None, dense=True)
    set.seq_len = 2
    set.sx = 1
    set.sy = 1
    set.change_image_size(128)
    print("dataset len:", len(set))
except Exception as e:
    print("Dataset loading error:", e)

# set.change_pixel_size(0.1)


[Lazy] Indexed 3 files, total usable samples: 34330
Image size changed to:  128
Origin offset changed to:  6.4
dataset len: 34330


In [3]:
# 데이터셋 분할 및 DataLoader 생성 (lazy loading 최적화)
# train_size = int(len(set) * 0.95)
# print(f"train_size: {train_size}")
# val_size = int(len(set) * 0)
# test_size = len(set) - (train_size + val_size)
# train_dataset, val_dataset, test_dataset = random_split(set, [train_size, val_size, test_size])

val_set = CenterSpeedDataset('/home/harry/sim_ws/src/f1tenth_gym_ros/Val_set', transform=None, dense=True)
val_set.seq_len = 2
val_set.sx = 1
val_set.sy = 1
val_set.change_image_size(128)

train_size = int(len(set))
print(f"train_size: {train_size}")
val_size = len(val_set)
print(f"val_size: {val_size}")

batch_size = 32
# 기존 preload 방식 DataLoader (주석처리)
# training_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# validation_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
# testing_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# lazy loading + DataLoader 최적화 파라미터 적용
training_loader = DataLoader(set, batch_size=batch_size, 
                             shuffle=True, num_workers=4, pin_memory=True, 
                             persistent_workers=True, prefetch_factor=4)

validation_loader = DataLoader(val_set, batch_size=batch_size, 
                               shuffle=False, num_workers=4, pin_memory=True, 
                               persistent_workers=True, prefetch_factor=4)

# testing_loader = DataLoader(test_dataset, batch_size=1, 
#                             shuffle=False, num_workers=4, pin_memory=True, 
#                             persistent_workers=True, prefetch_factor=4)


[Lazy] Indexed 3 files, total usable samples: 13021
Image size changed to:  128
Origin offset changed to:  6.4
train_size: 34330
val_size: 13021


In [4]:
# 모델, optimizer, loss 함수 정의
device = torch.device("cuda" if torch.cuda.is_available()else "cpu")
print(device)

net = CenterSpeedDense(image_size=128).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=5e-4) #weight_decay=1e-4)

print(net.input_channels)
print(net)

# is_free : LiDAR 기반으로 장애물이 없는 곳 -> 1, 장애물 있는 곳 -> 0
def dense_loss(output, gt_heatmap, gt_dense_data, is_free, alpha=0.9, decay=1):
    device = output.device
    gt_heatmap = gt_heatmap.to(device, dtype=torch.float32)
    gt_dense_data = gt_dense_data.to(device, dtype=torch.float32)
    loss = 0
    batch_size = output.shape[0]

    w = gt_heatmap
    # occupancy 채널: (batch, H, W)
    loss += (alpha * (1+w) * (output[:,0,:,:] - gt_heatmap)**2).sum()
    # dense 채널: (batch, 3, H, W) vs (batch, 3, H, W)
    loss += ((1-alpha) * (1+w).unsqueeze(1) * (output[:,1:,:,:] - gt_dense_data)**2).sum()
    return loss / batch_size
    
    # heat_loss = ((1+w) * (output[:,0]-gt_heatmap)**2).mean()
    # vec_loss  = (((1+w).unsqueeze(1)) * (output[:,1:]-gt_dense_data)**2).mean()
    # loss = alpha*heat_loss + (1-alpha)*vec_loss

    # return loss


cuda
4
CenterSpeedDense(
  (conv1): Conv2d(4, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (deconv1): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (deconv2): ConvTranspose2d(64, 4, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (bn4): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)


In [5]:
# --- EarlyStopping & ReduceLROnPlateau Scheduler 추가 ---
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.5):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# ReduceLROnPlateau 스케줄러 설정   엘알온플래토
from torch.optim.lr_scheduler import ReduceLROnPlateau

#factor = 0.5-> 학습률 절반으로 줄임, threshold 손실 개선 기준 
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10,threshold=0.5)

# EarlyStopping 인스턴스 생성
early_stopper = EarlyStopping(patience=15, min_delta=0.5)

In [8]:

from tqdm.auto import tqdm
import time

def train_and_validate(EPOCHS):
    losses = []
    val_losses = []
    epoch_durations = []
    start_all = time.time()

    for epoch in range(EPOCHS):
        print(f"Epoch: {epoch}")
        # net.to(device)
        net.train()
        running_loss = 0.0
        start_epoch = time.time()

        # ── Train 진행바 ──────────────────────────────────────────
        pbar = tqdm(training_loader, desc="Train", leave=False, dynamic_ncols=True)
        last = time.time()
        # print(start_epoch-last)
        for i, batch in enumerate(pbar, 1):
            inputs, gts, data_, dense_data, is_free = batch
            inputs     = inputs.to(device)
            gts        = gts.to(device)
            data_      = data_.to(device)
            # dense_data shape 맞추기 [B, H, W, 3] -> [B, 3, H, W]
            if dense_data.dim() == 4 and dense_data.shape[1] != 3:
                dense_data = dense_data.permute(0, 3, 1, 2)
            dense_data = dense_data.to(device)
            is_free    = is_free.to(device)

            optimizer.zero_grad(set_to_none=True)
            output = net(inputs)
            loss = dense_loss(output, gts, dense_data, is_free)
            
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # tqdm에 현재 배치 loss/ETA 표시
            now = time.time()
            batch_time = now - last
            last = now
            rem = len(training_loader) - i
            eta_batch = rem * batch_time
            pbar.set_postfix(loss=f"{loss.item():.4f}", eta=f"{eta_batch/60:.1f}m")

        avg_loss = running_loss / max(len(training_loader), 1)
        losses.append(avg_loss)

        # ── Validation 진행바(선택) ───────────────────────────────
        net.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            vpbar = tqdm(validation_loader, desc="Val", leave=False, dynamic_ncols=True)
            for batch in vpbar:
                inputs, gts, data_, dense_data, is_free = batch
                inputs     = inputs.to(device)
                gts        = gts.to(device)
                data_      = data_.to(device)
                # dense_data shape 맞추기 [B, H, W, 3] -> [B, 3, H, W]
                if dense_data.dim() == 4 and dense_data.shape[1] != 3:
                    dense_data = dense_data.permute(0, 3, 1, 2)
                dense_data = dense_data.to(device)
                is_free    = is_free.to(device)

                output = net(inputs)
                val_loss = dense_loss(output, gts, dense_data, is_free)

                # 디버깅용 출력
                # print("val input shape:", inputs.shape)
                # print("val gt shape:", gts.shape)
                # print("val dense_data shape:", dense_data.shape)
                # print("val is_free:", is_free)
                # print("val output min/max:", output.min().item(), output.max().item())
                # print("val loss:", val_loss.item())

                running_val_loss += val_loss.item()
                vpbar.set_postfix(loss=f"{val_loss.item():.4f}")

        avg_val_loss = running_val_loss / max(len(validation_loader), 1)
        val_losses.append(avg_val_loss)

        # --- ReduceLROnPlateau & EarlyStopping 적용 ---
        # scheduler.step(avg_val_loss)
        # early_stopper(avg_val_loss)
        # if early_stopper.early_stop:
        #     print(f"Early stopping at epoch {epoch+1}!")
        #     break

        # ── 10에폭마다 모델 저장 ───────────────────────────────
        SAVE_EVERY = 10
        if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == EPOCHS:
            save_path = f"/home/harry/ros2_ws/src/TinyCenterSpeed/src/trained_models/redbull_Damn_epoch_{epoch+1}.pt"
            torch.save(net.state_dict(), save_path)
            print(f"✅ 모델 저장 완료: {save_path}")

        # ── 에폭 ETA/로그 ────────────────────────────────────────
        epoch_sec = time.time() - start_epoch
        epoch_durations.append(epoch_sec)
        avg_epoch_sec = sum(epoch_durations) / len(epoch_durations)
        remaining_epochs = EPOCHS - (epoch + 1)
        eta_min = max(0.0, remaining_epochs * avg_epoch_sec) / 60.0

        if wandb.run is not None:
            wandb.log({
                "epoch": epoch,
                "train/loss": avg_loss,
                "val/loss": avg_val_loss,
                "time/epoch_sec": epoch_sec,
                "time/eta_minutes": eta_min
            }, commit=True)

        print(f'  train_loss={avg_loss:.6f} | val_loss={avg_val_loss:.6f} | epoch_sec={epoch_sec:.2f}s | ETA={eta_min:.1f} min')

    return losses, val_losses


In [9]:
# W&B 설정 및 학습 실행
wandb.init(project='CenterSpeed', name='0812_sx1_5e-4_Tlqkf_Damn', config={'batch_size': 32, 'lr': 5e-4, 'epochs': 100})
losses, val_losses = train_and_validate(EPOCHS=100)
wandb.finish()

Epoch: 0


Train:   0%|          | 0/1073 [00:00<?, ?it/s]

Val:   0%|          | 0/407 [00:00<?, ?it/s]

  train_loss=11490.648973 | val_loss=94557.480210 | epoch_sec=27.42s | ETA=45.2 min
Epoch: 1


Train:   0%|          | 0/1073 [00:00<?, ?it/s]

...

Train:   0%|          | 0/1073 [00:00<?, ?it/s]

Val:   0%|          | 0/407 [00:00<?, ?it/s]

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x739f0fbbeb30>> (for post_run_cell), with arguments args (<ExecutionResult object at 739f0fbbcc70, execution_count=9 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 739f0fbbfa60, raw_cell="# W&B 설정 및 학습 실행
wandb.init(project='CenterSpeed',.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/harry/ros2_ws/src/TinyCenterSpeed/src/train/train_CenterSpeed_wandb.ipynb#X10sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

In [ ]:
# 학습 곡선 시각화
plt.plot(losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training/Validation Loss Curve')
plt.legend()
plt.show()